# Health QA Colab Saved Model Prediction

Use this notebook when a fine-tuned model already exists in Drive and you only need to regenerate validation predictions and a Zindi submission.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/elvispreakerebi/health-qa-summative.git'
BRANCH = 'colab-llm-finetune'
REPO_DIR = '/content/health-qa-summative'
DRIVE_DATA_DIR = '/content/drive/MyDrive/mlt1-summative-datasets'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/health-qa-summative-outputs'

!rm -rf {REPO_DIR}
!git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!mkdir -p data/raw {DRIVE_OUTPUT_DIR}
!cp {DRIVE_DATA_DIR}/Train.csv data/raw/Train.csv
!cp {DRIVE_DATA_DIR}/Val.csv data/raw/Val.csv
!cp {DRIVE_DATA_DIR}/Test.csv data/raw/Test.csv
!python -m pip install -q -r requirements.txt
!python -m pip install -q -e .

In [ ]:
CONFIG = 'configs/colab_afriteva_v2_lora_full.yaml'
RUN_NAME = CONFIG.split('/')[-1].replace('.yaml', '')
MODEL_DIR = f'{DRIVE_OUTPUT_DIR}/{RUN_NAME}/final_model'
OUTPUT_DIR = f'outputs/{RUN_NAME}_predict'
print(CONFIG, MODEL_DIR, OUTPUT_DIR)

In [ ]:
!python -m health_qa.cli predict-generate --config {CONFIG} --model-dir {MODEL_DIR} --output-dir {OUTPUT_DIR}

In [ ]:
import pandas as pd
from health_qa.submission import validate_submission

metrics = pd.read_csv(f'{OUTPUT_DIR}/metrics.csv')
submission = pd.read_csv(f'{OUTPUT_DIR}/submission.csv')
validate_submission(submission)
display(metrics)
display(submission.head())
!mkdir -p {DRIVE_OUTPUT_DIR}/{RUN_NAME}_predict
!cp {OUTPUT_DIR}/metrics.csv {DRIVE_OUTPUT_DIR}/{RUN_NAME}_predict/metrics.csv
!cp {OUTPUT_DIR}/validation_predictions.csv {DRIVE_OUTPUT_DIR}/{RUN_NAME}_predict/validation_predictions.csv
!cp {OUTPUT_DIR}/submission.csv {DRIVE_OUTPUT_DIR}/{RUN_NAME}_predict/submission.csv
print('Saved submission to:', f'{DRIVE_OUTPUT_DIR}/{RUN_NAME}_predict/submission.csv')